# 0. Foundation

Builds the frozen files every section uses. The Producer runs this **once**; everyone else only uses the outputs:

- `clean_songs.csv`: one row per unique song, with its parent genre, a `split` column (train/test) and a `fold` column (0-4)
- `predictions/test_predictions.csv`: the final model's genre predictions on the test set

Frozen on Nov 20. Don't re-run or edit after that, so everyone's numbers stay identical.

In [1]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from common import FEATURES, GENRES, load, final_model

df = pd.read_csv("dataset.csv")

## Data cleaning: group sub-genres into 5 parent genres

In [2]:
broad_genre_map = {
    
    # Pop-ish music
    "pop": "pop",
    "power-pop": "pop",
    "synth-pop": "pop",
    "indie-pop": "pop",
    "k-pop": "pop",
    "j-pop": "pop",
    "pop-film": "pop",
    "children": "pop",
    "kids": "pop",
    "j-dance": "pop",
    "j-idol": "pop",
    "mandopop": "pop",
    "cantopop": "pop",

    # Rock and metal
    "rock": "rock",
    "alt-rock": "rock",
    "alternative": "other", #rock was my worst performing on everything so i went back and killed the more genre ambiguous subgenres
    "indie": "other",       #same as above
    "punk": "rock",
    "punk-rock": "rock",
    "psych-rock": "rock",
    "hard-rock": "rock",
    "rock-n-roll": "rock",
    "rockabilly": "rock",
    "grunge": "rock",
    "metal": "rock",
    "heavy-metal": "rock",
    "black-metal": "rock",
    "death-metal": "rock",
    "metalcore": "rock",
    "goth": "rock",
    "emo": "rock",
    "grindcore": "rock",

    # Hip-hop / R&B
    "hip-hop": "hiphop/rnb",
    "r-n-b": "hiphop/rnb",
    "soul": "hiphop/rnb",
    "funk": "hiphop/rnb",
    "groove": "other",
    

    # Electronic
    "edm": "electronic",
    "electronic": "electronic",
    "electro": "electronic",
    "house": "electronic",
    "deep-house": "electronic",
    "progressive-house": "electronic",
    "chicago-house": "electronic",
    "minimal-techno": "electronic",
    "techno": "electronic",
    "trance": "electronic",
    "dubstep": "electronic",
    "drum-and-bass": "electronic",
    "breakbeat": "electronic",
    "idm": "electronic",
    "club": "electronic",
    "dance": "electronic",
    "dancehall": "electronic",
    "garage": "electronic",
    "trip-hop": "electronic",

    # Latin
    "latin": "latin",
    "latino": "latin",
    "salsa": "latin",
    "samba": "latin",
    "pagode": "latin",
    "sertanejo": "latin",
    "mpb": "latin",
    "brazil": "latin",
    "forro": "latin",
    "reggaeton": "latin",
    "reggae": "latin",
    "tango": "latin",

    # Jazz/ Blues
    "jazz": "jazz-blues",
    "blues": "jazz-blues",
    
    # Country
    "bluegrass": "country",
    "folk": "country",
    "country": "country",
    "honky-tonk": "country",

    # Classical
    "classical": "classical",
    "opera": "classical",
    "piano": "classical",
    "ambient": "classical",
    "new-age": "classical",

    # other
    "afrobeat": "other",
    "indian": "other",
    "iranian": "other",
    "turkish": "other",
    "malay": "other",
    "world-music": "other",
    "french": "other",
    "german": "other",
    "spanish": "other",
    "swedish": "other",
    "acoustic": "other",
    "chill": "other",
    "happy": "other",
    "sad": "other",
    "sleep": "other",
    "study": "other",
    "party": "other",
    "romance": "other",
    "gospel": "other",
    "comedy": "other",
    "disco": "other",
    "disney": "other",
    "show-tunes": "other",
    "guitar": "other",
    "songwriter": "other",
    "singer-songwriter": "other"
}

df["broad_genre"] = df["track_genre"].map(broad_genre_map)
major_genres = ["pop", "rock", "hiphop/rnb", "electronic", "latin"]
df = df[df["broad_genre"].isin(major_genres)]

## Remove ambiguous and duplicate songs

Some songs are tagged under two different parent genres (for example both pop and rock), so their true genre is unclear. Drop them. Then keep one copy of every song, because the same song often appears several times.

In [3]:
song = df["artists"].astype(str).str.lower() + " | " + df["track_name"].astype(str).str.lower()
ambiguous = (df.groupby("track_id")["broad_genre"].transform("nunique") > 1) | (df.groupby(song)["broad_genre"].transform("nunique") > 1)
df = df[~ambiguous].drop_duplicates("track_id").drop_duplicates(["artists", "track_name"])
df["broad_genre"].value_counts()

broad_genre
electronic    13650
rock          11912
pop           10206
latin          7372
hiphop/rnb     1906
Name: count, dtype: int64

## Split by artist

20% of artists go to the test set, so no artist is on both sides, with each genre's share kept about the same. The training rows get 5 cross-validation folds, also grouped by artist.

In [4]:
artists = df["artists"].fillna("unknown")
_, test_rows = next(StratifiedGroupKFold(5, shuffle=True, random_state=42).split(df, df["broad_genre"], groups=artists))
df["split"] = "train"
df.iloc[test_rows, df.columns.get_loc("split")] = "test"

train = df["split"] == "train"
df["fold"] = -1
folds = StratifiedGroupKFold(5, shuffle=True, random_state=42).split(df[train], df.loc[train, "broad_genre"], groups=artists[train])
for k, (_, val_rows) in enumerate(folds):
    df.loc[df.index[train][val_rows], "fold"] = k

keep = ["track_id", "artists", "track_name", "track_genre", "broad_genre", "popularity"] + FEATURES + ["split", "fold"]
df[keep].to_csv("clean_songs.csv", index=False)
print(len(df), "unique songs |", train.sum(), "train |", (~train).sum(), "test")

45046 unique songs | 36037 train | 9009 test


## Final model predictions on the test set

In [5]:
d = load()
model = final_model().fit(d.X_train, d.y_train)

test = d.songs[d.songs["split"] == "test"][["track_id", "artists", "track_name", "broad_genre"]].copy()
test["predicted"] = model.predict(d.X_test)
proba = model.predict_proba(d.X_test)
for i, genre in enumerate(model.classes_):
    test[f"proba_{genre}"] = proba[:, i]
Path("predictions").mkdir(exist_ok=True)
test.to_csv("predictions/test_predictions.csv", index=False)
print("Test accuracy:", round((test["predicted"] == test["broad_genre"]).mean(), 3))
test.head()

Test accuracy: 0.64


,track_id,artists,track_name,broad_genre,predicted,proba_electronic,proba_hiphop/rnb,proba_latin,proba_pop,proba_rock
0,5E30LdtzQTGqRvNd7l6kG5,The Neighbourhood,Daddy Issues,rock,electronic,0.568408,0.023032,0.037308,0.214473,0.156780
2,2K7xn816oNHJZ0aVqdQsha,The Neighbourhood,Softcore,rock,electronic,0.350949,0.053233,0.067129,0.262920,0.265769
3,7zwn1eykZtZ5LODrf7c0tS,The Neighbourhood,You Get Me So High,rock,electronic,0.595679,0.041973,0.051678,0.140306,0.170364
4,7vguMCv8uVuZLiQJ156u3Z,Sam Tinnesz;Yacht Money,Play with Fire (feat. Yacht Money),rock,electronic,0.428363,0.063067,0.048956,0.228126,0.231489
7,6rrKbzJGGDlSZgLphopS49,The Score,Unstoppable,rock,rock,0.162324,0.006511,0.030395,0.182856,0.617915
